# 3. 编码注意机制

In [1]:
# 从标准库的 importlib.metadata 模块导入 version 函数
# 这个函数可以用来查询已安装 Python 包的版本号
from importlib.metadata import version

# 打印 PyTorch 框架的版本号
# version("torch") 会返回当前环境中安装的 torch 版本字符串
print("torch version:", version("torch"))

torch version: 2.3.1


## 3.3.1

> Step 1：计算未归一化的注意力分数

In [2]:
import torch

# 输入是一个 6×3 的张量
# 每一行代表一个词的 3 维嵌入向量
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your   (x^1)
        [0.55, 0.87, 0.66],  # journey (x^2)
        [0.57, 0.85, 0.64],  # starts (x^3)
        [0.22, 0.58, 0.33],  # with   (x^4)
        [0.77, 0.25, 0.10],  # one    (x^5)
        [0.05, 0.80, 0.55]   # step   (x^6)
    ]
)

In [3]:
# 选择第2个输入词元（"journey"）作为查询
query = inputs[1]  # 对应 x^(2)

# 初始化一个空张量来存储注意力分数
attn_scores_2 = torch.empty(inputs.shape[0])
print(inputs.shape)
attn_scores_2

torch.Size([6, 3])


tensor([0., 0., 0., 0., 0., 0.])

In [4]:
# 遍历所有输入词元，计算点积得到未归一化注意力分数
for i, x_i in enumerate(inputs):
    # 计算查询与每个输入向量的点积
    attn_scores_2[i] = torch.dot(x_i, query)
print(0.43 * 0.55 + 0.15 * 0.87 + 0.89 * 0.66)
attn_scores_2

0.9544


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

点积本质上是两个向量按元素相乘后求和的简写形式。

In [5]:
res = 0.
# 手动计算点积：按元素相乘后累加
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
# 使用 PyTorch 内置函数计算点积
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


>步骤 2：归一化注意力分数

对未归一化的注意力分数（ω）进行归一化，使它们的总和为 1。

In [6]:
# 简单归一化：分数除以所有分数的总和
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


 所有权重的和为 1，且每个权重代表对应输入词元的相对重要性。

然而在实践中，更常用且推荐的是 Softmax 函数进行归一化，它能更好地处理极端值，并且在训练时拥有更理想的梯度特性。

In [7]:
def softmax_naive(x):
    # Softmax 公式：对每个元素取指数后，除以所有元素指数的和
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


 因此在实际应用中，建议使用 PyTorch 内置的 softmax 函数，它经过了高度优化，能避免数值问题。

In [8]:
# 使用 PyTorch 内置的 Softmax 函数对注意力分数进行归一化
# dim=0 表示在第0个维度（即输入序列维度）上进行归一化
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


>步骤 3：将每个输入词元的嵌入向量 x(i) 与对应的注意力权重相乘，再将所有结果向量相加，得到上下文向量 z(2)。

In [9]:
# 选择第2个输入词元作为查询
query = inputs[1]  # 对应 "journey"

# 初始化上下文向量，维度与查询向量一致
context_vec_2 = torch.zeros(query.shape)

# 遍历所有输入词元，计算加权和
for i, x_i in enumerate(inputs):
    # 每个输入向量乘以对应的注意力权重，再累加到上下文向量
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


这个 3 维向量就是以 “journey” 为查询时得到的上下文向量，它融合了所有输入词元的信息。

# 总结

1. 注意力机制核心
- 动态为输入词元分配权重，让模型选择性关注关键信息
- 自注意力让序列中每个位置能与所有位置交互，捕捉全局依赖

2. 简化自注意力计算流程
- 点积计算未归一化注意力分数（衡量相似度）
- Softmax 归一化得到权重（总和为 1）
- 加权求和得到上下文向量（融合全局信息）

3. 关键技术点
- 点积是相似度计算的基础
- Softmax 保证权重可解释性与训练稳定性
- 上下文向量是输入向量的增强表示

# 3.3.2 为所有输入 token 计算注意力权重

将步骤 1 应用于所有两两元素，计算出未归一化的注意力分数矩阵：

In [10]:
attn_scores = torch.empty(6, 6) # 初始化一个 6×6 的空张量，用于存储注意力分数

for i, x_i in enumerate(inputs): # 遍历输入中的每个元素（作为“查询”）
    for j, x_j in enumerate(inputs): # 再次遍历输入中的每个元素（作为“键”）
        attn_scores[i][j] = torch.dot(x_i, x_j) # 计算两个输入向量的点积，作为它们的注意力分数

# 打印最终的注意力分数矩阵
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


我们可以通过矩阵乘法更高效地实现与上一步完全相同的结果：

In [ ]:
# 使用矩阵乘法直接计算所有两两输入的点积
attn_scores = inputs @ inputs.T # @ 是矩阵乘法运算符，等价于 torch.matmul(inputs, inputs.T)
# 打印注意力分数矩阵
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


对每一行进行归一化处理，让每行的数值之和为 1：

In [13]:
attn_weights = torch.softmax(attn_scores, dim=-1) # 对注意力分数的每一行做 Softmax 归一化，得到注意力权重
# 打印注意力权重矩阵
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


快速验证一下：每一行的数值之和确实为 1：

In [14]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print(row_2_sum)

1.0


In [16]:
print("All row sums:", attn_weights.sum(dim=-1))

All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


计算所有上下文向量：

In [18]:
all_context_vecs = attn_weights @ inputs # 用注意力权重对输入向量做加权求和，得到所有上下文向量
# 打印所有上下文向量
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [19]:
# 打印之前单独计算的第2个上下文向量
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


# 总结

1. 自注意力机制的核心三步流程
- 计算注意力分数：通过输入向量两两之间的点积（或矩阵乘法 inputs @ inputs.T），衡量 token 间的关联强度。
- 计算注意力权重：用 torch.softmax 对分数归一化，使每行权重之和为 1，得到每个 token 对其他 token 的关注度。
- 计算上下文向量：用注意力权重对输入向量加权求和（attn_weights @ inputs），得到融合全局信息的输出向量。
2. 矩阵乘法的高效性

    用矩阵乘法替代双重循环，可大幅提升计算效率，且结果完全一致。这是实现自注意力的标准做法。

3. 核心目标

    自注意力的最终目标是为每个输入 token 生成一个上下文向量，该向量能融合整个序列中与当前 token 最相关的信息，从而让模型更好地理解语义。